# Q3: Learning from Unlabeled Data

This experiment uses confidence-based pseudo-labeling to investigate whether unlabeled examples can improve our TF-IDF Logistic Regression model's validation performance.

In [1]:
import sys
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

import pandas as pd
from sklearn.base import clone

In [2]:
# locate project directory
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

sys.path.insert(0, str(project_dir))

from data_utils import load_data, split_data

# Load competition datasets
zip_path = project_dir / "data" / "cornell-tech-cs-5785-combined-2026-ai-vs-human-text.zip"

train, unlabeled, test = load_data(zip_path)

# Use the same validation split as Q1 and Q2
train_df, val_df = split_data(train)

print(f"Training examples: {len(train_df)}")
print(f"Validation examples: {len(val_df)}")
print(f"Unlabeled examples: {len(unlabeled)}")

Training examples: 8000
Validation examples: 2000
Unlabeled examples: 3000


## Supervised baseline

In [3]:
# same tuned model used in q1 + q2
supervised_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 3),
        min_df=5,
        max_features=50000
    )),
    ("logreg", LogisticRegression(
        C=10.0,
        max_iter=1000,
        random_state=42
    ))
])

# fit on labeled training data only
supervised_model.fit(train_df["text"], train_df["label"])

# evaluate on fixed validation set
val_probs = supervised_model.predict_proba(val_df["text"])[:, 1]
supervised_auc = roc_auc_score(val_df["label"], val_probs)

print(f"Supervised validation AUROC: {supervised_auc:.4f}")

Supervised validation AUROC: 0.9357


## Confidence-Based Pseudo-Labeling

We use the supervised model to predict labels for the unlabeled
dataset. To reduce the risk of adding incorrect labels, we only
select examples with predicted probabilities >= 0.9 or <= 0.1.

In [4]:
# predict AI probabilities for 3,000 unlabeled examples
unlabeled_probs = supervised_model.predict_proba(unlabeled["text"])[:, 1]

# select only high-confidence predictions
threshold = 0.9
confident_mask = (unlabeled_probs >= threshold) | (unlabeled_probs <= 1 - threshold)

# create dataset containing selected examples
pseudo_df = unlabeled.loc[confident_mask, ["text"]].copy()

# assign pseudo-labels
# 1 = AI, 0 = human
pseudo_df["label"] = (unlabeled_probs[confident_mask] >= 0.5).astype(int)

# summarize selected examples
print(f"Total unlabeled examples: {len(unlabeled)}")
print(f"Selected high-confidence examples: {len(pseudo_df)}")
print(f"Pseudo-labeled human examples: {(pseudo_df['label'] == 0).sum()}")
print(f"Pseudo-labeled AI examples: {(pseudo_df['label'] == 1).sum()}")


Total unlabeled examples: 3000
Selected high-confidence examples: 1224
Pseudo-labeled human examples: 607
Pseudo-labeled AI examples: 617


## Semi-Supervised Model

We combine the original labeled training data with the selected
pseudo-labeled examples and retrain a fresh Logistic Regression
model using the same hyperparameters.

In [5]:
# combine original labeled training data with pseudo-labeled examples
combined_train = pd.concat([
    train_df[["text", "label"]],
    pseudo_df[["text", "label"]]
], ignore_index=True)

print(f"Original labeled examples: {len(train_df)}")
print(f"Pseudo-labeled examples: {len(pseudo_df)}")
print(f"Combined training examples: {len(combined_train)}")

# create fresh model with the same hyperparameters
semi_model = clone(supervised_model)

# retrain using labeled + pseudo-labeled data
semi_model.fit(combined_train["text"], combined_train["label"])

# evaluate on the same untouched validation set
semi_val_probs = semi_model.predict_proba(val_df["text"])[:, 1]
semi_auc = roc_auc_score(val_df["label"], semi_val_probs)

# compare against supervised baseline
print(f"\nSupervised validation AUROC: {supervised_auc:.4f}")
print(f"Semi-supervised validation AUROC: {semi_auc:.4f}")
print(f"AUROC difference: {semi_auc - supervised_auc:+.4f}")

Original labeled examples: 8000
Pseudo-labeled examples: 1224
Combined training examples: 9224

Supervised validation AUROC: 0.9357
Semi-supervised validation AUROC: 0.9351
AUROC difference: -0.0006



## Results and Discussion

The supervised model achieved a validation AUROC of 0.9357 using 8,000 labeled examples. With a confidence threshold of 0.9, we
selected 1,224 of the 3,000 unlabeled examples, including 607 predicted human and 617 predicted AI passages.

After retraining on 9,224 examples, the semi-supervised model achieved a validation AUROC of 0.9351, a slight decrease of
0.0006. This suggests that confidence-based pseudo-labeling did not provide a meaningful improvement in this experiment.
Possible reasons include incorrect pseudo-labels or limited additional information from highly confident predictions.

The original validation set remained separate throughout training and pseudo-labeling.
